<a href="https://colab.research.google.com/github/Lparedes14/msis-assignments/blob/retail-relabel/Business_Problem_About_Retail_Relabeling_using_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Business Problem About Retail Relabeling using Logistic Regression

Group: Luis Paredes - Javier Brito

## Part 1 - Feature engineering and EDA

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

df = pd.read_csv("retail_relabel.csv")

In [6]:
df["CompAvg"] = df[["Competitor_BigBoxDepot", "Competitor_ValueMart",
                    "Competitor_QuickShop"]].mean(axis=1)
df["GapPct"] = 100 * (df["StorePrice"] - df["CompAvg"]) / df["CompAvg"]
df["AbsGapPct"] = df["GapPct"].abs()
df["LogUnits"] = np.log(df["WeeklyUnitsSold"])
df["MarginPct"] = 100 * (df["StorePrice"] - df["UnitCost"]) / df["StorePrice"]

print(df["Relabel"].value_counts(normalize=True).rename("share"), "\n")
print(df.groupby("ElectronicShelfLabel")["Relabel"].mean().round(3), "\n")

Relabel
0    0.6
1    0.4
Name: share, dtype: float64 

ElectronicShelfLabel
0    0.339
1    0.541
Name: Relabel, dtype: float64 



In [12]:
print(df.groupby("Category")["Relabel"].mean().round(3), "\n")

Category
Apparel         0.390
Electronics     0.440
Grocery         0.412
Household       0.378
PersonalCare    0.383
Name: Relabel, dtype: float64 



## Part 2 - A naive model

In [7]:
m1 = smf.logit("Relabel ~ GapPct + UnitCost + RelabelCost", data=df).fit(disp=0)
print("MODEL 1 (signed gap)  pseudo-R2=%.3f  AIC=%.1f" % (m1.prsquared, m1.aic))

MODEL 1 (signed gap)  pseudo-R2=0.077  AIC=1871.0


### Interpretation — why the signed gap is the wrong functional form

`GapPct` is signed, so the model is forced to fit **a single slope** across positive and negative values. But the owner's actual decision rule is **not monotonic** in the signed gap — it's symmetric around zero:

- **Store priced 10% above** competitors → losing sales to the competition → the owner wants to **cut** the price.
- **Store priced 10% below** competitors → leaving margin on the table, but demand is fine → the owner wants to **raise** the price.

Both cases are misalignments that justify a relabel, but a single signed coefficient can only capture "more positive gap → more relabeling" or the reverse — it **cannot represent** that a large misalignment in either direction increases the probability of a relabel.

Using `AbsGapPct` instead is the correct approach because it recodes the decision as depending on the **magnitude** of misalignment, matching how a retail manager actually treats over- and under-pricing: as mirror-image problems requiring the same action — fix the price.

Model 1's much lower pseudo-R² and higher AIC/BIC versus Model 2 confirm this functional-form mismatch **empirically**, not just theoretically.

## Part 3: Correct model (absolute gap + full drivers)

In [8]:
m2 = smf.logit(
    "Relabel ~ AbsGapPct + CostChangePct + LogUnits + RelabelCost "
    "+ DaysSinceLastChange + UnitCost", data=df).fit(disp=0)
print("MODEL 2 (abs gap)     pseudo-R2=%.3f  AIC=%.1f\n" % (m2.prsquared, m2.aic))
print(m2.summary2().tables[1].round(4), "\n")

or_table = pd.DataFrame({
    "OddsRatio": np.exp(m2.params),
    "CI_low": np.exp(m2.conf_int()[0]),
    "CI_high": np.exp(m2.conf_int()[1]),
}).round(3)
print("Odds ratios:\n", or_table, "\n")

MODEL 2 (abs gap)     pseudo-R2=0.275  AIC=1477.8

                      Coef.  Std.Err.        z  P>|z|  [0.025  0.975]
Intercept           -4.2772    0.5673  -7.5397  0.000 -5.3891 -3.1653
AbsGapPct            0.1778    0.0109  16.2451  0.000  0.1563  0.1992
CostChangePct        0.1276    0.0168   7.5839  0.000  0.0946  0.1606
LogUnits             0.4324    0.1311   3.2996  0.001  0.1756  0.6893
RelabelCost         -0.8589    0.0852 -10.0849  0.000 -1.0258 -0.6920
DaysSinceLastChange  0.0040    0.0006   6.3688  0.000  0.0028  0.0053
UnitCost             0.0004    0.0037   0.1017  0.919 -0.0069  0.0077 

Odds ratios:
                      OddsRatio  CI_low  CI_high
Intercept                0.014   0.005    0.042
AbsGapPct                1.195   1.169    1.220
CostChangePct            1.136   1.099    1.174
LogUnits                 1.541   1.192    1.992
RelabelCost              0.424   0.359    0.501
DaysSinceLastChange      1.004   1.003    1.005
UnitCost                 1.000   0.99

In [13]:
print("MODEL 1 (signed gap)  pseudo-R2=%.3f  AIC=%.1f  BIC=%.1f" % (m1.prsquared, m1.aic, m1.bic))
print("MODEL 2 (abs gap)     pseudo-R2=%.3f  AIC=%.1f  BIC=%.1f\n" % (m2.prsquared, m2.aic, m2.bic))

MODEL 1 (signed gap)  pseudo-R2=0.077  AIC=1871.0  BIC=1892.3
MODEL 2 (abs gap)     pseudo-R2=0.275  AIC=1477.8  BIC=1515.0



### Interpretation — Model 2 odds ratios, in plain English for the owner

**AbsGapPct (OR ≈ 1.20):** each additional percentage point of price misalignment versus competitors — in either direction — raises the odds of a relabel by about 20%. This is the core competitive-repricing signal: the further a shelf tag drifts from the market, the more likely it gets corrected.

**RelabelCost (OR ≈ 0.42):** each additional dollar of relabel cost cuts the odds of relabeling by roughly 58%. This is the menu-cost effect showing up exactly where economic theory predicts it — the cost of the action itself (paper tag labor/materials vs. near-zero for an ESL) is a real deterrent, separate from how misaligned the price actually is.

**LogUnits (OR ≈ 1.54):** a one-log-point increase in weekly units sold (roughly a 2.7x increase in volume) raises the odds of relabeling by about 54%. Fast-moving SKUs get repriced more readily because a mispriced fast mover bleeds margin (or lost sales) every day it sits wrong — the per-day cost of leaving the tag alone is higher, so the relabel pays for itself sooner.


## Part 4: Out-of-sample evaluation

In [9]:
feats = ["AbsGapPct", "CostChangePct", "LogUnits", "RelabelCost",
         "DaysSinceLastChange", "UnitCost"]
X_tr, X_te, y_tr, y_te = train_test_split(
    df[feats], df["Relabel"], test_size=0.30, random_state=7,
    stratify=df["Relabel"])
train = X_tr.copy(); train["Relabel"] = y_tr
m3 = smf.logit("Relabel ~ " + " + ".join(feats), data=train).fit(disp=0)
p_hat = m3.predict(X_te)
y_hat = (p_hat >= 0.5).astype(int)
print("Confusion matrix:\n", confusion_matrix(y_te, y_hat))
print("Accuracy: %.3f  AUC: %.3f\n" % (accuracy_score(y_te, y_hat),
                                       roc_auc_score(y_te, p_hat)))


Confusion matrix:
 [[233  37]
 [ 64 116]]
Accuracy: 0.776  AUC: 0.853




#### Why CostChangePct matters but the level of UnitCost does not

`CostChangePct` is highly significant (p < 0.001) while raw `UnitCost` is not (p = 0.919). This is exactly what menu-cost theory predicts: a relabel is triggered by a **change** that has pushed the current price out of line — a recent supplier cost increase erodes margin and forces a repricing decision. The **static level** of a product's unit cost, by contrast, carries no information about whether today's price is currently misaligned; a 50 dollar item and a 2 dollar item are equally likely to need relabeling if neither has experienced a recent cost shock. The model is picking up the **trigger** (a change happened), not a fixed characteristic of the SKU (its price point) — which is the correct economic signal.

---

#### Model comparison and deployment choice

Model 2 has a substantially higher pseudo-R² and lower AIC/BIC than Model 1 — the gaps are large, not marginal, well beyond the ~2–6 point rule-of-thumb threshold considered "strong evidence" in AIC/BIC comparisons. Since BIC penalizes added parameters more heavily than AIC, and Model 2 still wins on BIC despite having more covariates, this confirms the extra predictors are earning their keep rather than overfitting.

**I would deploy Model 2**: it is both better specified theoretically (menu costs plus the correct, symmetric gap measure) and better supported by the data.

## Part 5: Economic threshold

Expected weekly value of relabeling one unit of |gap| on a fast mover vs the relabel cost: students compute break-even p* = C / (C + B) style reasoning.

In [10]:
test = X_te.copy()
test["p_hat"] = p_hat
test["ExpectedWeeklyLoss"] = (test["AbsGapPct"] / 100) \
    * df.loc[test.index, "StorePrice"] * df.loc[test.index, "WeeklyUnitsSold"]
test["NetBenefit"] = test["ExpectedWeeklyLoss"] - test["RelabelCost"]
print("Share of SKUs where one week of mispricing already exceeds the "
      "relabel cost: %.3f" % (test["NetBenefit"] > 0).mean())


Share of SKUs where one week of mispricing already exceeds the relabel cost: 0.987


In [11]:
# ---------- Marginal effects ----------
print("\nAverage marginal effects:\n", m2.get_margeff().summary())


Average marginal effects:
         Logit Marginal Effects       
Dep. Variable:                Relabel
Method:                          dydx
At:                           overall
                         dy/dx    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
AbsGapPct               0.0286      0.001     25.849      0.000       0.026       0.031
CostChangePct           0.0205      0.003      8.128      0.000       0.016       0.025
LogUnits                0.0695      0.021      3.342      0.001       0.029       0.110
RelabelCost            -0.1381      0.012    -11.463      0.000      -0.162      -0.115
DaysSinceLastChange     0.0006   9.72e-05      6.680      0.000       0.000       0.001
UnitCost             6.103e-05      0.001      0.102      0.919      -0.001       0.001
